# 二元樹 Binary Tree

1. 用 Class 建立二元搜尋樹（BST）與插入節點
2. 前序、中序、後序拜訪
3. 用「陣列」表示完全二元樹：`tree[2p]`、`tree[2p+1]`
4. 如何決定唯一樹
5. 實戰：完全二元樹層和、FBI Binary Tree（ZeroJudge）


## 1. 用 Class 建立節點與二元搜尋樹

二元搜尋樹（BST）規則：左子樹 < 樹根 < 右子樹。


In [ ]:
class Node:
    def __init__(self, data):
        self.right = None
        self.left = None
        self.data = data

def Insert(root, data):
    if root is None:
        return Node(data)
    if data < root.data:
        root.left = Insert(root.left, data)
    else:
        root.right = Insert(root.right, data)
    return root

root = None
for v in [5, 3, 8, 1, 4, 7, 9]:
    root = Insert(root, v)
print("建樹完成")


## 2. 三種拜訪方式

| 拜訪 | 順序 |
|---|---|
| 前序 PreOrder | 樹根 → 左 → 右 |
| 中序 InOrder | 左 → 樹根 → 右 |
| 後序 PostOrder | 左 → 右 → 樹根 |

**BST 的中序拜訪 = 由小到大排序**。


In [ ]:
def PreOrder(root, l):
    if root:
        l.append(str(root.data))
        PreOrder(root.left, l)
        PreOrder(root.right, l)

def InOrder(root, l):
    if root:
        InOrder(root.left, l)
        l.append(str(root.data))
        InOrder(root.right, l)

def PostOrder(root, l):
    if root:
        PostOrder(root.left, l)
        PostOrder(root.right, l)
        l.append(str(root.data))

for name, f in [("前序", PreOrder), ("中序", InOrder), ("後序", PostOrder)]:
    l = []
    f(root, l)
    print(name, " ".join(l))


## 3. 用陣列表示完全二元樹

樹根放在 index 1，對任何節點 `p`：

- 左子節點：`2 * p`
- 右子節點：`2 * p + 1`
- 父節點：`p // 2`

不用寫 Class，競賽時更快更不容易出錯。


In [ ]:
#            A(1)
#           /    \
#         B(2)   C(3)
#        /  \
#      D(4) E(5)
tree = [''] * 16
tree[1] = 'A'
tree[2] = 'B'
tree[3] = 'C'
tree[4] = 'D'
tree[5] = 'E'

def preorder(p):
    if p < len(tree) and tree[p] != '':
        print(tree[p], end='')
        preorder(2 * p)
        preorder(2 * p + 1)

def inorder(p):
    if p < len(tree) and tree[p] != '':
        inorder(2 * p)
        print(tree[p], end='')
        inorder(2 * p + 1)

def postorder(p):
    if p < len(tree) and tree[p] != '':
        postorder(2 * p)
        postorder(2 * p + 1)
        print(tree[p], end='')

preorder(1); print("  ← 前序")
inorder(1);  print("  ← 中序")
postorder(1); print("  ← 後序")


## 4. 如何決定唯一樹

- 中序表示式 + 前序表示式 → **可以**決定唯一樹
- 中序表示式 + 後序表示式 → **可以**決定唯一樹
- 前序表示式 + 後序表示式 → **無法**決定唯一樹

反例：「根 1，只有左子 2，2 只有左子 3」與「根 1，只有右子 2，2 只有右子 3」……
兩棵不同的樹，前序都是 `1 2 3`，後序都是 `3 2 1`。


In [ ]:
# 由前序 + 中序重建樹，輸出後序
def build_postorder(preorder, inorder):
    if not preorder:
        return ""
    root = preorder[0]
    k = inorder.index(root)
    left  = build_postorder(preorder[1:k+1], inorder[:k])
    right = build_postorder(preorder[k+1:], inorder[k+1:])
    return left + right + root

print(build_postorder("ABDEC", "DBEAC"))   # 應輸出 DEBCA


## 5. 完全二元樹：層和問題

【題目描述】給定一棵包含 $N$ 個節點的完全二元樹，
按從上到下、從左到右的順序節點數字依次是 $A_1, \dots, A_N$。
求「節點數字之和最大」的階層；若有多層同為最大，輸出最小的階層（樹根是第 1 層）。

【範例輸入】

```text
7
1 2 3 4 5 6 7
```

【範例輸出】

```text
3
```

（第 1 層和 = 1，第 2 層和 = 5，第 3 層和 = 22）


In [ ]:
sample = "7\n1 2 3 4 5 6 7"
data = sample.split("\n")

def solve(data):
    n_ = int(data[0])
    a = list(map(int, data[1].split()))
    best_level, best_sum = 0, None
    level, start = 1, 0
    while start < n_:
        cnt = min(2 ** (level - 1), n_ - start)   # 這一層的節點數
        s = sum(a[start:start + cnt])
        if best_sum is None or s > best_sum:
            best_sum, best_level = s, level
        start += cnt
        level += 1
    return best_level

print(solve(data))


## 6. FBI Binary Tree

題目網址：<https://zerojudge.tw/ShowProblem?problemid=f606>

【題目描述】由 0 和 1 組成的字串可分成三種：
全部是 0 的稱為 **B** 串、全部是 1 的稱為 **I** 串、同時包含 0 和 1 的稱為 **F** 串。

給定長度 $2^N$ 的 01 字串，建樹規則：
每個節點代表一段字串（型態為 B/I/F），把字串對半分，左半是左子樹、右半是右子樹，
直到長度為 1。請輸出這棵樹的**後序拜訪**。

【範例輸入】

```text
3
10001011
```

【範例輸出】

```text
IBFBBBFIBFIIIFF
```


In [ ]:
sample = "3\n10001011"
data = sample.split("\n")

N = int(data[0])
s = data[1]

tree = [''] * (4 * len(s))

def node_type(seg):
    if '0' not in seg:
        return 'I'
    if '1' not in seg:
        return 'B'
    return 'F'

def build(p, L, R):
    tree[p] = node_type(s[L:R + 1])
    if L == R:
        return
    mid = (L + R) // 2
    build(2 * p, L, mid)
    build(2 * p + 1, mid + 1, R)

def postorder_str(p):
    if tree[p] == '':
        return ""
    return postorder_str(2 * p) + postorder_str(2 * p + 1) + tree[p]

build(1, 0, len(s) - 1)
print(postorder_str(1))    # 應輸出 IBFBBBFIBFIIIFF


## 7. 練習題

用「前序 + 中序」重建樹並輸出後序：

前序：`FBADCE`、中序：`ABDCEF`

期望輸出：`AECDBF`


In [ ]:
# 練習作答區
pre, ino = "FBADCE", "ABDCEF"


### 參考答案

In [ ]:
pre, ino = "FBADCE", "ABDCEF"
print(build_postorder(pre, ino))
